# 🛡️ PhishGuard AI - Phishing URL Detection Model Training

This notebook demonstrates the complete machine learning workflow for building a Phishing URL Detection Classifier. The workflow includes:
- **Data Loading & Exploration**
- **Data Preprocessing & Cleaning**
- **Feature Selection**
- **Model Training & Hyperparameter Tuning** (using Random Forest and Logistic Regression)
- **Model Comparison & Evaluation** (using classification metrics like Accuracy, Precision, Recall, F1-Score, and ROC-AUC)
- **Results Visualization** (Confusion Matrices, ROC Curves, and Feature Importance)

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import json
import os

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix, roc_curve
)

%matplotlib inline
sns.set_theme(style="whitegrid")

## 2. Load the Dataset
We load the phishing and legitimate website dataset, checking its structure and label distribution.

In [ ]:
dataset_path = 'dataset/Phishing_Legitimate_full.csv'
df = pd.read_csv(dataset_path)

print(f"Dataset Shape: {df.shape}")
df.head()

In [ ]:
# Check class distribution (1 = Phishing, 0 = Legitimate)
print("Class Distribution:")
print(df['CLASS_LABEL'].value_counts(normalize=True))

## 3. Preprocessing & Cleaning
We perform the following tasks:
- Drop the unique record identifier `id` column to prevent overfitting.
- Fill any missing values with 0.
- Drop highly imbalanced and redundant threshold features (`NoHttps`, `SubdomainLevelRT`, `UrlLengthRT`, etc.).

In [ ]:
# Drop ID column if present
if 'id' in df.columns:
    df = df.drop(columns=['id'])

# Handle missing values
df = df.fillna(0)

# Drop highly imbalanced / redundant features
drop_cols = [
    'NoHttps',
    'SubdomainLevelRT',
    'UrlLengthRT',
    'PctExtResourceUrlsRT',
    'AbnormalExtFormActionR',
    'ExtMetaScriptLinkRT',
    'PctExtNullSelfRedirectHyperlinksRT'
]
df = df.drop(columns=drop_cols, errors='ignore')

# Separate features and target label
target = 'CLASS_LABEL'
X = df.drop(columns=[target])
y = df[target]
feature_names = list(X.columns)

print(f"Cleaned Features Shape: {X.shape}")

## 4. Train/Test Split
We split the data into 80% training and 20% testing sets using a fixed seed (`random_state=42`) for reproducibility.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

## 5. Feature Scaling
We scale features using `StandardScaler` specifically for the Logistic Regression model, since distance-based and linear models are sensitive to feature scales. Random Forest is tree-based and scale-invariant, so it does not require scaled data.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 6. Model Training & Parameter Tuning

### 6.1 Model 1: Random Forest Classifier
We use `GridSearchCV` to optimize the Random Forest hyperparameters and run 5-fold cross-validation.

In [ ]:
rf_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 20, 30],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
}

base_rf = RandomForestClassifier(random_state=42, n_jobs=-1)
rf_grid_search = GridSearchCV(base_rf, rf_param_grid, cv=3, scoring='accuracy', n_jobs=-1, verbose=1)
rf_grid_search.fit(X_train, y_train)

rf_clf = rf_grid_search.best_estimator_
print(f"Random Forest Best Parameters: {rf_grid_search.best_params_}")
print(f"Random Forest Best CV Accuracy: {rf_grid_search.best_score_:.4f}")

In [ ]:
# Run 5-fold cross-validation with the best estimator
rf_cv_scores = cross_val_score(rf_clf, X_train, y_train, cv=5, scoring='accuracy')
print(f"Mean RF CV Accuracy: {rf_cv_scores.mean():.4f} (+/- {rf_cv_scores.std() * 2:.4f})")

### 6.2 Model 2: Logistic Regression
We optimize and train our Logistic Regression model using scaled training features.

In [ ]:
lr_param_grid = {
    'C': [0.01, 0.1, 1.0, 10.0],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear']
}

base_lr = LogisticRegression(random_state=42, max_iter=1000)
lr_grid_search = GridSearchCV(base_lr, lr_param_grid, cv=3, scoring='accuracy', n_jobs=-1, verbose=1)
lr_grid_search.fit(X_train_scaled, y_train)

lr_clf = lr_grid_search.best_estimator_
print(f"Logistic Regression Best Parameters: {lr_grid_search.best_params_}")
print(f"Logistic Regression Best CV Accuracy: {lr_grid_search.best_score_:.4f}")

In [ ]:
# Run 5-fold cross-validation for Logistic Regression
lr_cv_scores = cross_val_score(lr_clf, X_train_scaled, y_train, cv=5, scoring='accuracy')
print(f"Mean LR CV Accuracy: {lr_cv_scores.mean():.4f} (+/- {lr_cv_scores.std() * 2:.4f})")

## 7. Model Evaluation & Comparison

In [ ]:
y_pred_rf = rf_clf.predict(X_test)
y_prob_rf = rf_clf.predict_proba(X_test)[:, 1]

y_pred_lr = lr_clf.predict(X_test_scaled)
y_prob_lr = lr_clf.predict_proba(X_test_scaled)[:, 1]

metrics_rf = {
    'Accuracy': accuracy_score(y_test, y_pred_rf),
    'Precision': precision_score(y_test, y_pred_rf),
    'Recall': recall_score(y_test, y_pred_rf),
    'F1-Score': f1_score(y_test, y_pred_rf),
    'ROC-AUC': roc_auc_score(y_test, y_prob_rf)
}

metrics_lr = {
    'Accuracy': accuracy_score(y_test, y_pred_lr),
    'Precision': precision_score(y_test, y_pred_lr),
    'Recall': recall_score(y_test, y_pred_lr),
    'F1-Score': f1_score(y_test, y_pred_lr),
    'ROC-AUC': roc_auc_score(y_test, y_prob_lr)
}

comparison_df = pd.DataFrame([metrics_rf, metrics_lr], index=['Random Forest', 'Logistic Regression'])
comparison_df.round(4)

In [ ]:
print("Random Forest Classification Report:")
print(classification_report(y_test, y_pred_rf))
print("\nLogistic Regression Classification Report:")
print(classification_report(y_test, y_pred_lr))

## 8. Visualizations

### 8.1 Confusion Matrix Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

cm_rf = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Legitimate', 'Phishing'], yticklabels=['Legitimate', 'Phishing'],
            annot_kws={'size': 12, 'weight': 'bold'})
axes[0].set_title(f"Random Forest Confusion Matrix\n(Accuracy: {metrics_rf['Accuracy']:.4f})", fontsize=13, pad=10)
axes[0].set_xlabel('Predicted Label', fontsize=11)
axes[0].set_ylabel('True Label', fontsize=11)

cm_lr = confusion_matrix(y_test, y_pred_lr)
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Oranges', ax=axes[1],
            xticklabels=['Legitimate', 'Phishing'], yticklabels=['Legitimate', 'Phishing'],
            annot_kws={'size': 12, 'weight': 'bold'})
axes[1].set_title(f"Logistic Regression Confusion Matrix\n(Accuracy: {metrics_lr['Accuracy']:.4f})", fontsize=13, pad=10)
axes[1].set_xlabel('Predicted Label', fontsize=11)
axes[1].set_ylabel('True Label', fontsize=11)

plt.tight_layout()
plt.show()

### 8.2 ROC Curve Comparison

In [ ]:
plt.figure(figsize=(8.5, 6.5))
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)

plt.plot(fpr_rf, tpr_rf, label=f"Random Forest (AUC = {metrics_rf['ROC-AUC']:.4f})", color='blue', lw=2.5)
plt.plot(fpr_lr, tpr_lr, label=f"Logistic Regression (AUC = {metrics_lr['ROC-AUC']:.4f})", color='darkorange', lw=2.5)
plt.plot([0, 1], [0, 1], 'k--', label='Random Guessing (AUC = 0.5000)', lw=1.5)

plt.xlabel('False Positive Rate (FPR)', fontsize=11)
plt.ylabel('True Positive Rate (TPR)', fontsize=11)
plt.title('Receiver Operating Characteristic (ROC) Curve Comparison', fontsize=13, pad=15)
plt.legend(loc='lower right', fontsize=10)
plt.grid(True, alpha=0.3, linestyle='--')
plt.show()

### 8.3 Feature Importance (Random Forest)

In [ ]:
importances = rf_clf.feature_importances_
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(11, 7))
sns.barplot(data=importance_df.head(15), x='Importance', y='Feature', hue='Feature', legend=False, palette='viridis')
plt.title('Top 15 Most Important Features (Random Forest)', fontsize=13, pad=15)
plt.xlabel('Feature Importance Score', fontsize=11)
plt.ylabel('Features', fontsize=11)
plt.grid(True, alpha=0.2, axis='x', linestyle='--')
plt.show()